# 71 Veeva Link Phase 1 features (P1 to P4)

Implements tasks P1.1 to P4 of `70_veeva_link_phase1_working_plan.md`. Builds the Veeva T1 and T2 block at the NBA 3.1 grain **`NPI_NUMBER` x `COHORT_ID`** from the Phase 0 outputs (`52_` to `56_`).

* **Cohorts:** NBA 3.1 C1 to C22 (cohort months 2025-01 to 2026-10) and NBA 3.0 C01 to C26 (2023-04 to 2025-05; used only for the E2 vs E3 leakage test). Offset 7: cut-off = month end + 6 days, `COHORT_ID = YYYY-MM_offset7`. Rules copied from `51_digital_nba_phase1_features`, which mirrors `00_cohort_config` (to confirm, input I2). Using offset 7 for the NBA 3.0 cohorts is an assumption (input I4).
* **As-of rule:** T1 windows end at min(cohort cut-off, table data cut-off) and are 5 years for trials / publications and 3 years for events / payments. Payment program year `Y` counts once published (30 June of `Y + 1`).
* **Speed:** cohorts that share the same window ends share one build. Each distinct window set (a **signature**) is built once, staged, and copied to its cohorts.
* **Missing is not zero:** `NOT_IN_VEEVA` NPIs are null on every feature except the coverage flags.
* **T3 is not joined here** (P1-D4).

**Outputs** under `OUT/features_phase1/`: `t1_t2/COHORT_ID=<id>/`, `t1_static/` (E2), `feature_manifest/`, `cohort_windows/`, `checks/run=<ts>/`, `schema_lock/`.

In [0]:
from pyspark.sql import functions as F
from functools import reduce
from dateutil.relativedelta import relativedelta
import pandas as pd, datetime as dt, calendar, json, os

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAPSHOT_ID = "load=2025-06-30"
SNAP, CFG, ACT = f"{OUT}snapshot/{SNAPSHOT_ID}/", f"{OUT}config/", f"{OUT}activity/"
P1 = f"{OUT}features_phase1/"
STAGE = f"{P1}_stage/"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
rd = lambda t: spark.read.parquet(f"{SNAP}{t}")

# Optional main specialty source (input I3): a table or path with NPI_NUMBER and SPECIALTY. None = not available yet
SPECIALTY_SOURCE = None

spine = spark.read.parquet(f"{OUT}spine/veeva_npi_se_map")
CUTOFF = {r.table: dt.date.fromisoformat(r.data_cutoff) for r in spark.read.parquet(f"{SNAP}_manifest").collect()}
WIN = {r.families: r.months for r in spark.read.parquet(CFG + "windows_v1").collect()}
M_TRIAL_PUB, M_EVENT_PAY = WIN["trials; publications"], WIN["events; payments"]
CHILD_CUTOFF = min(CUTOFF[t] for t in ["clinical_trial", "publication", "medical_event", "payment"])
COMPETITORS = ["astrazeneca", "boehringer", "lilly", "janssen", "novo nordisk"]
print("table cut-offs:", {t: str(CUTOFF[t]) for t in ["clinical_trial", "publication", "medical_event", "payment", "scientific_expert"]},
      "| windows:", M_TRIAL_PUB, M_EVENT_PAY)

In [0]:
OFFSET_DAY = 7
NBA31 = (dt.date(2025, 1, 1), dt.date(2026, 10, 1))   # C1 to C22
NBA30 = (dt.date(2023, 4, 1), dt.date(2025, 5, 1))    # C01 to C26 (leakage test only)

def cohort_cutoff(cm):
    """Last day (inclusive) of cohort month cm: month end + (offset - 1) days."""
    return dt.date(cm.year, cm.month, calendar.monthrange(cm.year, cm.month)[1]) + dt.timedelta(days=OFFSET_DAY - 1)

def cohort_month_of(d):
    return (d - dt.timedelta(days=OFFSET_DAY - 1)).replace(day=1)

def months(a, b):
    out, m = [], a
    while m <= b:
        out.append(m); m += relativedelta(months=1)
    return out

# self-test (same cases as 00_cohort_config / 51_digital_nba_phase1_features)
for d, exp in [("2025-01-07", "2025-01-01"), ("2025-01-06", "2024-12-01"), ("2025-02-06", "2025-01-01"),
               ("2025-02-07", "2025-02-01"), ("2025-10-04", "2025-09-01")]:
    assert str(cohort_month_of(dt.date.fromisoformat(d))) == exp, (d, exp)
for cm in months(*NBA30) + months(*NBA31):
    assert cohort_month_of(cohort_cutoff(cm)) == cm and cohort_month_of(cohort_cutoff(cm) + dt.timedelta(days=1)) == cm + relativedelta(months=1)
assert len(months(*NBA31)) == 22 and len(months(*NBA30)) == 26

def windows_for(cut):
    """(start exclusive, end inclusive) per family, capped at the table cut-off; payment program years."""
    w = {}
    for fam, t, m in [("trials", "clinical_trial", M_TRIAL_PUB), ("publications", "publication", M_TRIAL_PUB), ("events", "medical_event", M_EVENT_PAY)]:
        end = min(cut, CUTOFF[t])
        w[fam] = ((pd.Timestamp(end) - pd.DateOffset(months=m)).date(), end)
    pend = min(cut, CUTOFF["payment"])
    last_y = pend.year - 1 if pend >= dt.date(pend.year, 6, 30) else pend.year - 2
    w["pay_years"] = tuple(range(last_y - M_EVENT_PAY // 12 + 1, last_y + 1))
    return w

rows = []
for cm in sorted(set(months(*NBA30)) | set(months(*NBA31))):
    cut, w = cohort_cutoff(cm), windows_for(cohort_cutoff(cm))
    sets = [s for s, r in [("nba_3_0", NBA30), ("nba_3_1", NBA31)] if r[0] <= cm <= r[1]]
    rows.append({"COHORT_ID": f"{cm:%Y-%m}_offset{OFFSET_DAY}", "cohort_month": cm, "cohort_cutoff": cut, "cohort_set": ",".join(sets),
                 "trials_start": w["trials"][0], "trials_end": w["trials"][1], "pubs_start": w["publications"][0], "pubs_end": w["publications"][1],
                 "events_start": w["events"][0], "events_end": w["events"][1], "pay_years": ",".join(map(str, w["pay_years"])),
                 "veeva_window_end": min(cut, CHILD_CUTOFF),
                 "veeva_data_age_months": max(0, (cut.year - CHILD_CUTOFF.year) * 12 + cut.month - CHILD_CUTOFF.month - (cut.day < CHILD_CUTOFF.day))})
cw = pd.DataFrame(rows)
sig_cols = ["trials_start", "trials_end", "pubs_start", "pubs_end", "events_start", "events_end", "pay_years"]
cw["sig"] = cw.groupby(sig_cols, sort=False).ngroup()
cw["pubs_left_cut"] = (cw.pubs_start < dt.date(2019, 1, 1)).astype(int)   # publication data starts 2019 (plan R3)
spark.createDataFrame(cw.astype({c: str for c in cw.columns if c not in ("sig", "veeva_data_age_months", "pubs_left_cut")})) \
     .write.mode("overwrite").parquet(P1 + "cohort_windows")
print("cohorts:", len(cw), "| NBA 3.1:", cw.cohort_set.str.contains("nba_3_1").sum(), "| NBA 3.0:", cw.cohort_set.str.contains("nba_3_0").sum(),
      "| distinct window signatures:", cw.sig.nunique())
display(cw[["COHORT_ID", "cohort_set", "cohort_cutoff", "trials_end", "pubs_start", "events_end", "pay_years", "sig", "veeva_data_age_months", "pubs_left_cut"]])

In [0]:
dup = spine.filter("NPI IS NOT NULL").groupBy("NPI").agg(F.count("*").alias("n_providers"), F.collect_list("PROVIDER_ID").alias("providers"))
dup_log = dup.filter("n_providers > 1").toPandas()
# keep one row per NPI (lowest PROVIDER_ID); the same NPI always maps to the same SE_ID, so values are identical
npi_spine = (spine.filter("NPI IS NOT NULL")
             .withColumn("rk", F.row_number().over(__import__("pyspark.sql.window", fromlist=["Window"]).Window.partitionBy("NPI").orderBy("PROVIDER_ID")))
             .filter("rk = 1").drop("rk")
             .select(F.col("NPI").alias("NPI_NUMBER"), "PROVIDER_ID", "SE_ID", "IN_VEEVA", "BEST_DECILE", "VEEVA_COVERAGE_STATUS"))
npi_spine.write.mode("overwrite").parquet(STAGE + "npi_spine")
npi_spine = spark.read.parquet(STAGE + "npi_spine")
matched = npi_spine.filter("IN_VEEVA = 1").select("SE_ID").distinct()
N_NPI = npi_spine.count()
se_conflict = dup.filter("n_providers > 1").join(spine.select("NPI", "SE_ID"), "NPI").groupBy("NPI").agg(F.countDistinct("SE_ID").alias("n")).filter("n > 1").count()
print("providers dropped (no NPI):", spine.filter("NPI IS NULL").count(), "| NPIs on 2+ providers:", len(dup_log),
      "| of those with different SE_IDs:", se_conflict, "| NPI rows:", N_NPI, "| matched experts:", matched.count())
display(npi_spine.groupBy("VEEVA_COVERAGE_STATUS").count())

In [0]:
IND = ["hf", "ckd", "t1d"]
FEATURE_SPEC = [
    ("veeva_coverage_status", "T2", "coverage", "NOT_IN_VEEVA / IN_VEEVA_UNSCORED / SCORED"),
    ("veeva_has_profile", "T2", "coverage", "1 if matched to SCIENTIFIC_EXPERT on NPI"),
    ("veeva_has_inscope_decile", "T2", "coverage", "1 if any in-scope decile (dictionary v1)"),
    ("veeva_n_inscope_themes", "T2", "coverage", "distinct dictionary themes with a decile"),
    ("veeva_trial_any_5y", "T1", "trials", "any trial active in window"),
    ("veeva_trial_log_n_5y", "T1", "trials", "log1p distinct trials in window"),
    ("veeva_trial_pi_flag_5y", "T1", "trials", "PI on any trial in window"),
    ("veeva_trial_n_inscope_5y", "T1", "trials", "distinct in-scope trials in window"),
    *[(f"veeva_trial_n_inscope_{i}_5y", "T1", "trials", f"distinct trials tagged {i.upper()} in window") for i in IND],
    ("veeva_trial_max_phase_inscope_5y", "T1", "trials", "max phase 1-4 of in-scope trials, 0 if none"),
    ("veeva_trial_active_flag", "T1", "trials", "trial running at window end"),
    ("veeva_trial_finerenone_flag_5y", "T1", "trials", "trial name tagged finerenone"),
    ("veeva_trial_mra_flag_5y", "T1", "trials", "trial name tagged MRA or finerenone"),
    ("veeva_trial_sglt2_flag_5y", "T1", "trials", "trial name tagged SGLT2"),
    ("veeva_trial_cardiorenal_n_5y", "T1", "trials", "distinct trials tagged CKD / cardiorenal / landmark / finerenone / MRA / SGLT2 / ERA"),
    ("veeva_pub_any_5y", "T1", "publications", "any publication in window"),
    ("veeva_pub_log_n_5y", "T1", "publications", "log1p publications in window"),
    ("veeva_pub_senior_author_flag_5y", "T1", "publications", "first or last author in window"),
    ("veeva_pub_n_inscope_5y", "T1", "publications", "in-scope publications in window"),
    *[(f"veeva_pub_n_inscope_{i}_5y", "T1", "publications", f"publications tagged {i.upper()} in window") for i in IND],
    ("veeva_pub_months_since_last", "T1", "publications", "months from window end to last publication, null if none"),
    ("veeva_event_any_3y", "T1", "events", "any event in window"),
    ("veeva_event_podium_flag_3y", "T1", "events", "speaker / chair / panelist / moderator in window"),
    ("veeva_event_poster_flag_3y", "T1", "events", "poster presenter in window"),
    ("veeva_event_n_inscope_3y", "T1", "events", "distinct in-scope events in window"),
    *[(f"veeva_event_n_inscope_{i}_3y", "T1", "events", f"distinct events tagged {i.upper()} in window") for i in IND],
    ("veeva_pay_any_3y", "T1", "payments", "any payment, published program years in window"),
    ("veeva_pay_log_total_3y", "T1", "payments", "log1p total USD"),
    ("veeva_pay_excl_food_log_3y", "T1", "payments", "log1p total USD excluding food and beverage"),
    ("veeva_pay_n_companies_3y", "T1", "payments", "distinct paying companies"),
    ("veeva_pay_speaker_consult_flag_3y", "T1", "payments", "speaker / faculty / consulting payment"),
    ("veeva_pay_competitor_flag_3y", "T1", "payments", "payment from a competitor company (not Bayer)"),
    ("veeva_pay_research_flag_3y", "T1", "payments", "research payment"),
    ("veeva_pay_bayer_flag_3y", "T1", "payments", "Bayer payment (separate flag only, D3)"),
    ("veeva_expert_type", "T2", "profile", "EXPERT_TYPE"),
    ("veeva_is_md_do", "T2", "profile", "MD or DO in DEGREE_1..3"),
    ("veeva_is_phd", "T2", "profile", "PhD in DEGREE_1..3"),
    ("veeva_is_academic", "T2", "profile", "affiliation looks academic"),
    ("veeva_emerging_expert", "T2", "profile", "EMERGING_EXPERT = Y"),
    ("veeva_n_editorial_boards", "T2", "profile", "editorial board roles"),
    ("veeva_n_society_roles", "T2", "profile", "scientific / professional association roles"),
    ("veeva_kidney_society_flag", "T2", "profile", "association name mentions kidney / nephrology / renal"),
    ("veeva_n_sponsors", "T2", "profile", "distinct sponsoring companies"),
    ("veeva_specialty_fallback", "T2", "profile", "Veeva SPECIALTY_1 only where the NBA 3.1 specialty is missing (input I3)"),
    ("veeva_industry_engagement", "T1", "composite", "mean z of pay_log_total, pay_n_companies, pay_speaker_consult (matched NPIs of the cohort)"),
    ("veeva_network_breadth", "T1", "composite", "mean z of pub_log_n, event_any, n_society_roles (matched NPIs of the cohort)"),
]
COLS = [f[0] for f in FEATURE_SPEC]
T1_COLS = [f[0] for f in FEATURE_SPEC if f[1] == "T1"]
NON_ZERO_FILL = {"veeva_coverage_status", "veeva_has_profile", "veeva_has_inscope_decile", "veeva_expert_type",
                 "veeva_pub_months_since_last", "veeva_specialty_fallback", "veeva_industry_engagement", "veeva_network_breadth"}
KEYS = ["NPI_NUMBER", "PROVIDER_ID"]
META = ["cohort_set", "veeva_window_end", "veeva_data_age_months", "veeva_snapshot_id"]
assert len(COLS) == len(set(COLS))
print("features:", len(COLS), "| T1:", len(T1_COLS), "| T2:", len(COLS) - len(T1_COLS))

In [0]:
fd = spark.read.parquet(CFG + "focus_dictionary_v1")
def scored(t, keys, dcol):
    k = fd.filter(f"source_table = '{t}'").select(F.col("focus_name").alias("FOCUS_NAME"), F.col("cl_type").alias("TYPE"), "theme")
    k = k if t == "clinical_leader" else k.drop("TYPE")
    return (rd(t).join(matched, "SE_ID").join(F.broadcast(k), keys)
              .select("SE_ID", "theme", F.col(dcol).cast("int").alias("D")).filter("D BETWEEN 1 AND 10"))
themes = (scored("clinical_leader", ["FOCUS_NAME", "TYPE"], "SE_DECILE").unionByName(scored("focus_area", ["FOCUS_NAME"], "FOCUS_DECILE"))
          .unionByName(scored("digital_reach_focus_area", ["FOCUS_NAME"], "SE_DECILE"))
          .groupBy("SE_ID").agg(F.countDistinct("theme").alias("veeva_n_inscope_themes")))

se = rd("scientific_expert").join(matched, "SE_ID")
deg = F.upper(F.concat_ws("|", "DEGREE_1", "DEGREE_2", "DEGREE_3"))
prof = se.select("SE_ID", F.col("EXPERT_TYPE").alias("veeva_expert_type"),
                 deg.rlike(r"(^|\|)(MD|DO)(\||$)").cast("int").alias("veeva_is_md_do"),
                 deg.rlike(r"(^|\|)PHD(\||$)").cast("int").alias("veeva_is_phd"),
                 (F.col("EMERGING_EXPERT") == "Y").cast("int").alias("veeva_emerging_expert"),
                 F.col("SPECIALTY_1").alias("VEEVA_SPECIALTY_1"))
acad = (rd("address").join(matched, "SE_ID").select("SE_ID", F.lower(F.concat_ws(" ", "AFFILIATION", "DEPARTMENT")).alias("aff"))
        .groupBy("SE_ID").agg(F.max(F.col("aff").rlike("universit|college|school of medicine|academic").cast("int")).alias("veeva_is_academic")))
assoc = (rd("association").join(matched, "SE_ID").groupBy("SE_ID").agg(
    F.sum((F.col("ASSOCIATION_TYPE") == "Editorial Board").cast("int")).alias("veeva_n_editorial_boards"),
    F.sum(F.col("ASSOCIATION_TYPE").isin("Scientific Association", "Professional Association").cast("int")).alias("veeva_n_society_roles"),
    F.max(F.lower(F.concat_ws(" ", "ASSOCIATION_NAME", "ASSOCIATION_ORIGINAL_NAME")).rlike("kidney|nephrolog|renal").cast("int")).alias("veeva_kidney_society_flag")))
spons = rd("company_sponsorship").join(matched, "SE_ID").groupBy("SE_ID").agg(F.countDistinct("COMPANY_NAME").alias("veeva_n_sponsors"))
static_se = prof.join(acad, "SE_ID", "left").join(assoc, "SE_ID", "left").join(spons, "SE_ID", "left").join(themes, "SE_ID", "left")
static_se.write.mode("overwrite").parquet(STAGE + "static_se")
static_se = spark.read.parquet(STAGE + "static_se")

# Specialty fallback (P3): only where the main NBA 3.1 specialty is missing
if SPECIALTY_SOURCE:
    main_spec = (spark.table(SPECIALTY_SOURCE) if "." in SPECIALTY_SOURCE and not SPECIALTY_SOURCE.startswith("/") else spark.read.load(SPECIALTY_SOURCE))
    main_spec = main_spec.select(F.col("NPI_NUMBER").cast("string"), F.col("SPECIALTY").alias("MAIN_SPECIALTY"))
else:
    main_spec = None
print("profile rows:", static_se.count(), "| with Veeva SPECIALTY_1:", static_se.filter("VEEVA_SPECIALTY_1 IS NOT NULL").count(),
      "| main specialty source:", SPECIALTY_SOURCE or "not set (input I3): veeva_specialty_fallback stays null")

In [0]:
trials, pubs, events = (spark.read.parquet(ACT + t) for t in ["trials", "publications", "events"])
CARDIO = ["TAG_CKD", "TAG_CARDIORENAL", "TAG_LANDMARK_TRIAL", "TAG_FINERENONE", "TAG_MRA", "TAG_SGLT2", "TAG_ERA"]

# payments pre-aggregated once to SE x program year x company (cuts 38M rows)
(spark.read.parquet(ACT + "payments").withColumn("AMOUNT", F.expr("try_cast(AMOUNT AS DOUBLE)"))
   .groupBy("SE_ID", F.col("PROGRAM_YEAR").cast("int").alias("PY"), "COMPANY_NAME")
   .agg(F.sum("AMOUNT").alias("AMT"), F.sum(F.when(F.col("FOOD_FLAG") == 0, F.col("AMOUNT")).otherwise(F.lit(0.0))).alias("AMT_NOFOOD"),
        F.max("SPEAKER_CONSULT_FLAG").alias("SPK"), F.max("RESEARCH_FLAG").alias("RES"), F.max("BAYER_FLAG").alias("BAY"))
   .withColumn("COMP", (F.lower("COMPANY_NAME").rlike("|".join(COMPETITORS)) & (F.col("BAY") == 0)).cast("int"))
   .write.mode("overwrite").parquet(STAGE + "pay_sy"))
pay_sy = spark.read.parquet(STAGE + "pay_sy")

def t1_features(ts, te, ps, pe, es, ee, yrs):
    """T1 per SE_ID for one window set. Returns (DataFrame, look-ahead maxima)."""
    tw = trials.filter((F.col("START_DATE") <= F.lit(te)) & (F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(ts))))
    tr = tw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_trial_any_5y"), F.log1p(F.countDistinct("CLINICAL_TRIAL_ID")).alias("veeva_trial_log_n_5y"),
        F.max("PI_FLAG").alias("veeva_trial_pi_flag_5y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_n_inscope_5y"),
        *[F.countDistinct(F.when(F.col(f"TAG_{i.upper()}") == 1, F.col("CLINICAL_TRIAL_ID"))).alias(f"veeva_trial_n_inscope_{i}_5y") for i in IND],
        F.coalesce(F.max(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("PHASE_NUM"))), F.lit(0)).alias("veeva_trial_max_phase_inscope_5y"),
        F.max((F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(te))).cast("int")).alias("veeva_trial_active_flag"),
        F.max("TAG_FINERENONE").alias("veeva_trial_finerenone_flag_5y"),
        F.max(F.greatest("TAG_MRA", "TAG_FINERENONE")).alias("veeva_trial_mra_flag_5y"),
        F.max("TAG_SGLT2").alias("veeva_trial_sglt2_flag_5y"),
        F.countDistinct(F.when(F.greatest(*CARDIO) == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_cardiorenal_n_5y"))

    pw = pubs.filter((F.col("DATE") > F.lit(ps)) & (F.col("DATE") <= F.lit(pe)))
    pb = pw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pub_any_5y"), F.log1p(F.count("*")).alias("veeva_pub_log_n_5y"),
        F.max("SENIOR_AUTHOR_FLAG").alias("veeva_pub_senior_author_flag_5y"), F.sum("TAG_ANY_INSCOPE").alias("veeva_pub_n_inscope_5y"),
        *[F.sum(f"TAG_{i.upper()}").alias(f"veeva_pub_n_inscope_{i}_5y") for i in IND])
    last = (pubs.filter(F.col("DATE") <= F.lit(pe)).groupBy("SE_ID")
                .agg(F.floor(F.months_between(F.lit(pe), F.max("DATE"))).cast("int").alias("veeva_pub_months_since_last")))

    ew = events.filter((F.col("START_DATE") > F.lit(es)) & (F.col("START_DATE") <= F.lit(ee)))
    ev = ew.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_event_any_3y"), F.max("PODIUM_FLAG").alias("veeva_event_podium_flag_3y"),
        F.max("POSTER_FLAG").alias("veeva_event_poster_flag_3y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("MEDICAL_EVENT_ID"))).alias("veeva_event_n_inscope_3y"),
        *[F.countDistinct(F.when(F.col(f"TAG_{i.upper()}") == 1, F.col("MEDICAL_EVENT_ID"))).alias(f"veeva_event_n_inscope_{i}_3y") for i in IND])

    pw2 = pay_sy.filter(F.col("PY").isin(list(yrs)))
    py = pw2.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pay_any_3y"), F.log1p(F.sum("AMT")).alias("veeva_pay_log_total_3y"),
        F.log1p(F.sum("AMT_NOFOOD")).alias("veeva_pay_excl_food_log_3y"), F.countDistinct("COMPANY_NAME").alias("veeva_pay_n_companies_3y"),
        F.max("SPK").alias("veeva_pay_speaker_consult_flag_3y"), F.max("COMP").alias("veeva_pay_competitor_flag_3y"),
        F.max("RES").alias("veeva_pay_research_flag_3y"), F.max("BAY").alias("veeva_pay_bayer_flag_3y"))

    look = {"max_trial_start": tw.agg(F.max("START_DATE")).first()[0], "max_pub_date": pw.agg(F.max("DATE")).first()[0],
            "max_event_start": ew.agg(F.max("START_DATE")).first()[0], "max_pay_year": pw2.agg(F.max("PY")).first()[0]}
    out = matched
    for p in [tr, pb, last, ev, py]:
        out = out.join(p, "SE_ID", "left")
    return out, look

def zmean(df, cols, name):
    st = df.filter("veeva_has_profile = 1").agg(*[F.mean(c).alias(f"m_{c}") for c in cols], *[F.stddev(c).alias(f"s_{c}") for c in cols]).collect()[0]
    return df.withColumn(name, sum((F.col(c) - st[f"m_{c}"]) / st[f"s_{c}"] for c in cols) / len(cols))

def assemble(t1_se):
    """NPI grain: coverage + T1 + T2 + composites, with the missing rule."""
    se_df = t1_se.join(static_se, "SE_ID", "left")
    h = (npi_spine.join(se_df, "SE_ID", "left")
            .withColumn("veeva_coverage_status", F.col("VEEVA_COVERAGE_STATUS"))
            .withColumn("veeva_has_profile", F.col("IN_VEEVA"))
            .withColumn("veeva_has_inscope_decile", F.col("BEST_DECILE").isNotNull().cast("int")))
    if main_spec is not None:
        h = (h.join(main_spec, "NPI_NUMBER", "left")
              .withColumn("veeva_specialty_fallback", F.when(F.col("MAIN_SPECIALTY").isNull(), F.col("VEEVA_SPECIALTY_1"))))
    else:
        h = h.withColumn("veeva_specialty_fallback", F.lit(None).cast("string"))
    for c in [c for c in COLS if c not in NON_ZERO_FILL]:
        h = h.withColumn(c, F.when(F.col("IN_VEEVA") == 1, F.coalesce(F.col(c), F.lit(0))))
    h = zmean(h, ["veeva_pay_log_total_3y", "veeva_pay_n_companies_3y", "veeva_pay_speaker_consult_flag_3y"], "veeva_industry_engagement")
    h = zmean(h, ["veeva_pub_log_n_5y", "veeva_event_any_3y", "veeva_n_society_roles"], "veeva_network_breadth")
    return h.select(*KEYS, *COLS)

In [0]:
import time
look_rows = []
sigs = cw.drop_duplicates("sig").sort_values("sig")
for r in sigs.itertuples():
    t0 = time.time()
    t1_se, look = t1_features(r.trials_start, r.trials_end, r.pubs_start, r.pubs_end, r.events_start, r.events_end,
                              tuple(int(y) for y in r.pay_years.split(",")))
    assemble(t1_se).write.mode("overwrite").parquet(f"{STAGE}sig={r.sig}")
    look_rows.append({"sig": r.sig, **look, "seconds": round(time.time() - t0)})
look_df = pd.DataFrame(look_rows)
print("signatures built:", len(look_df), "| total minutes:", round(look_df.seconds.sum() / 60, 1))

# Static T1 for E2 (P1-D6): whole snapshot, no window; suffix _all
all_years = tuple(r[0] for r in pay_sy.select("PY").distinct().collect())
t1_all, look_all = t1_features(dt.date(1900, 1, 1), CHILD_CUTOFF, dt.date(1900, 1, 1), CUTOFF["publication"],
                               dt.date(1900, 1, 1), CUTOFF["medical_event"], all_years)
stat = assemble(t1_all).select(*KEYS, *[F.col(c).alias(c.replace("_5y", "_all").replace("_3y", "_all").replace("months_since_last", "months_since_last_all")) for c in T1_COLS])
stat.withColumn("veeva_snapshot_id", F.lit(SNAPSHOT_ID)).write.mode("overwrite").parquet(P1 + "t1_static")
print("t1_static rows:", spark.read.parquet(P1 + "t1_static").count(), "| payment program years used:", sorted(all_years))

In [0]:
from functools import reduce
stage_all = reduce(lambda a, b: a.unionByName(b), [spark.read.parquet(f"{STAGE}sig={s}").withColumn("sig", F.lit(int(s))) for s in sigs.sig])
nd = stage_all.agg(*[F.countDistinct(c).alias(c) for c in COLS]).toPandas().T[0]
WAITING = {"veeva_specialty_fallback"} if main_spec is None else set()   # kept in the schema while input I3 is open
constant = [c for c in COLS if nd[c] <= 1 and c not in WAITING]
KEEP = [c for c in COLS if c not in constant]
print("constant features dropped:", constant or "none", "| kept:", len(KEEP))

cmap = spark.createDataFrame(cw[["COHORT_ID", "sig", "cohort_set", "veeva_window_end", "veeva_data_age_months"]].astype({"veeva_window_end": str}))
final = (stage_all.join(F.broadcast(cmap), "sig")
         .withColumn("veeva_window_end", F.to_date("veeva_window_end")).withColumn("veeva_snapshot_id", F.lit(SNAPSHOT_ID))
         .select(*KEYS, "COHORT_ID", *KEEP, *META))
final.write.mode("overwrite").partitionBy("COHORT_ID").parquet(P1 + "t1_t2")

spec = pd.DataFrame(FEATURE_SPEC, columns=["feature", "tier", "family", "rule"])
spec["indication"] = spec.feature.str.extract(r"_(hf|ckd|t1d)_")[0].fillna("kerendia_total")
spec["kept"] = spec.feature.isin(KEEP)
spec["status"] = ["dropped_constant" if f in constant else ("waiting_I3" if f in WAITING else "kept") for f in spec.feature]
spec["E1"] = spec.family.eq("coverage"); spec["E2"] = spec.E1 | spec.tier.eq("T1"); spec["E3"] = spec.E2; spec["E4"] = True
spec["position"] = range(1, len(spec) + 1)
spec["snapshot_id"], spec["run_ts"] = SNAPSHOT_ID, RUN_TS
spark.createDataFrame(spec).write.mode("overwrite").parquet(P1 + "feature_manifest")
# count partitions through Spark (os.listdir on the volume can fail on serverless with 'Required key not available')
print("written cohorts:", spark.read.parquet(P1 + "t1_t2").select("COHORT_ID").distinct().count())

In [0]:
out = spark.read.parquet(P1 + "t1_t2")
exp_cols = KEYS + KEEP + META + ["COHORT_ID"]   # partition column comes last on read
checks = []
def add(k, name, value, expected, ok, note=""):
    checks.append({"check": k, "name": name, "value": str(value), "expected": str(expected), "pass": bool(ok), "note": note})

# K1 grain
g = out.groupBy("COHORT_ID").agg(F.count("*").alias("n"), F.countDistinct("NPI_NUMBER").alias("d")).toPandas()
add("K1", "one row per NPI per cohort", f"{g.n.min()}..{g.n.max()} rows, {len(g)} cohorts", f"{N_NPI} rows x {len(cw)} cohorts",
    len(g) == len(cw) and (g.n == N_NPI).all() and (g.d == N_NPI).all(),
    "Comparison with the NBA 3.1 claims NPI count not run: claims cohort output not readable yet")
# K2 fixed schema: manifest order now, and same as the previous run (schema_lock)
lock_p = P1 + "schema_lock/columns.json"
prev = json.load(open(lock_p)) if os.path.exists(lock_p) else None
add("K2", "columns = manifest order", len(out.columns), len(exp_cols), out.columns == exp_cols)
add("K2", "columns = previous run", "first run" if prev is None else (prev == out.columns), True if prev else "lock written",
    prev is None or prev == out.columns, "first run writes the lock" if prev is None else "")
os.makedirs(P1 + "schema_lock", exist_ok=True); json.dump(out.columns, open(lock_p, "w"))
# K3 no look-ahead
la = look_df.merge(sigs[["sig", "trials_end", "pubs_end", "events_end", "pay_years"]], on="sig")
bad = la[(pd.to_datetime(la.max_trial_start) > pd.to_datetime(la.trials_end)) | (pd.to_datetime(la.max_pub_date) > pd.to_datetime(la.pubs_end)) |
         (pd.to_datetime(la.max_event_start) > pd.to_datetime(la.events_end)) | (la.max_pay_year > la.pay_years.str.split(",").str[-1].astype(int))]
add("K3", "latest activity used <= window end", f"{len(bad)} signatures over", 0, len(bad) == 0)
# K4 missing is not zero
leak = out.filter("veeva_has_profile = 0").select(sum(F.col(c).isNotNull().cast("int") for c in KEEP if c not in NON_ZERO_FILL)).first()[0]
add("K4", "non-null T1/T2 values for NOT_IN_VEEVA", leak, 0, leak == 0)
# K5 cohort 2025-02 vs 56_ anchor 2025-10-23 (both capped at 2025-03-06)
ref = spark.read.parquet(OUT + "features/t1_t2/anchor=2025-10-23")
common = [c for c in KEEP if c in ref.columns and c not in ("veeva_coverage_status", "veeva_expert_type")]
j = out.filter("COHORT_ID = '2025-02_offset7'").select("PROVIDER_ID", *common).alias("a").join(ref.select("PROVIDER_ID", *common).alias("b"), "PROVIDER_ID")
tol = lambda c: 1e-3 if c in ("veeva_industry_engagement", "veeva_network_breadth") else 1e-9
mm = j.select(*[F.sum((~(F.abs(F.col(f"a.{c}") - F.col(f"b.{c}")) <= tol(c)) & ~(F.col(f"a.{c}").isNull() & F.col(f"b.{c}").isNull())).cast("int")).alias(c) for c in common]).toPandas().T[0]
add("K5", "cohort 2025-02 = 56_ anchor 2025-10-23", f"{int(mm.sum())} mismatching values in {len(common)} columns, {j.count()} rows", 0,
    mm.sum() == 0, "; ".join(f"{c}: {int(v)}" for c, v in mm.items() if v > 0))
# K6 cohort identity: share of NPIs whose T1 values change vs the previous cohort
# composites are re-standardised per cohort, so they move for every matched NPI; keep them out of the identity hash
t1k = [c for c in KEEP if c in T1_COLS and c not in ("veeva_industry_engagement", "veeva_network_breadth")]
h = out.select("NPI_NUMBER", "COHORT_ID", F.xxhash64(*t1k).alias("h"))
cw_sorted = cw.sort_values("cohort_month").reset_index(drop=True)
pairs = spark.createDataFrame(pd.DataFrame({"COHORT_ID": cw_sorted.COHORT_ID[1:].values, "PREV": cw_sorted.COHORT_ID[:-1].values}))
chg = (h.join(pairs, "COHORT_ID").alias("x").join(h.alias("y"), (F.col("x.PREV") == F.col("y.COHORT_ID")) & (F.col("x.NPI_NUMBER") == F.col("y.NPI_NUMBER")))
        .groupBy(F.col("x.COHORT_ID").alias("COHORT_ID")).agg(F.avg((F.col("x.h") != F.col("y.h")).cast("double")).alias("share_changed")).toPandas())
chg = chg.merge(cw[["COHORT_ID", "cohort_set"]], on="COHORT_ID").sort_values("COHORT_ID")
late31 = chg[chg.COHORT_ID > "2025-02_offset7"]
add("K6", "T1 unchanged for NBA 3.1 cohorts after 2025-02", f"max share changed {late31.share_changed.max():.4f}", 0, late31.share_changed.max() == 0,
    f"NBA 3.0 cohorts: mean share changed {chg[chg.COHORT_ID < '2025-02_offset7'].share_changed.mean():.4f}")
# K7 constant features
add("K7", "constant features dropped", len(constant), "listed", True, ", ".join(constant))

ck = pd.DataFrame(checks).assign(run_ts=RUN_TS, snapshot_id=SNAPSHOT_ID)
spark.createDataFrame(ck).write.mode("overwrite").parquet(f"{P1}checks/run={RUN_TS}")
display(ck)
print("ALL CHECKS PASS:", bool(ck["pass"].all()))
display(chg)

In [0]:
num = [c for c in KEEP if c not in ("veeva_coverage_status", "veeva_expert_type", "veeva_specialty_fallback")]
show = ["2023-04_offset7", "2024-06_offset7", "2025-01_offset7", "2025-02_offset7", "2026-10_offset7"]
s = (out.filter(F.col("COHORT_ID").isin(show) & (F.col("veeva_has_profile") == 1)).groupBy("COHORT_ID")
        .agg(*[F.round(F.mean(c), 4).alias(c) for c in num]).toPandas().set_index("COHORT_ID").T)
display(s.reset_index().rename(columns={"index": "feature"}))
display(look_df)
if len(dup_log):
    display(dup_log)